In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
!ls /kaggle/input/q1-stage-3-2026/PlantVillage/test/

In [ ]:
# Write your code here
import os
import glob
from torch.utils.data import Dataset
from PIL import Image
import os
from torchvision.datasets import ImageFolder
from torchvision import transforms
import matplotlib.pyplot as plt
import numpy as np
import torch

from torch.utils.data import DataLoader


class PotatoDiseaseDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform

        # 1. Auto-generate class dictionary (or define manually)
        # Sort ensures 0=Apple, 1=Banana consistently
        self.classes = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])
        self.class_labels = {cls_name: i for i, cls_name in enumerate(self.classes)}

        self.image_paths = []
        self.labels = []

        # 2. Collect Paths
        for class_name, label in self.class_labels.items():
            paths = glob.glob(f"{root_dir}/{class_name}/*.*") # Matches .jpg, .png etc
            self.image_paths.extend(paths)
            self.labels.extend([label] * len(paths))

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # 3. Load & Process
        image = Image.open(self.image_paths[idx]).convert("RGB")
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.RandomRotation(15),
    transforms.ToTensor()
])
test_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])

train_dir = os.path.join(path,"PlantVillage", "train")
test_dir = os.path.join(path,"PlantVillage", "test")

train_dataset = PotatoDiseaseDataset(root_dir=train_dir, transform=transform)
test_dataset = PotatoDiseaseDataset(root_dir=test_dir,transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)


import random
import numpy as np

# Create a function to visualize samples
def visualize_samples(dataset, num_samples=8, title="Dataset Samples"):
    """
    Visualize random samples from a dataset.

    Args:
        dataset: PyTorch Dataset object
        num_samples: Number of samples to display
        title: Title for the plot
    """
    # Select random indices
    indices = random.sample(range(len(dataset)), num_samples)

    # Calculate grid size
    cols = 4
    rows = (num_samples + cols - 1) // cols

    # Create the plot
    fig, axes = plt.subplots(rows, cols, figsize=(12, 3 * rows))
    axes = axes.flatten()

    for i, idx in enumerate(indices):
        # Get image and label
        image, label = dataset[idx]
        # Convert tensor to numpy for display
        if isinstance(image, torch.Tensor):
            image = image.permute(1, 2, 0).numpy()  # CHW -> HWC
        # Get class name
        class_name = dataset.classes[label]
        # Display image
        axes[i].imshow(image)
        axes[i].set_title(f"{class_name}\n(Label: {label})", fontsize=10)
        axes[i].axis('off')
    # Hide any unused subplots
    for i in range(num_samples, len(axes)):
        axes[i].axis('off')

    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualize_samples(train_dataset)



In [ ]:
# Write your code here
import torch.nn as nn
import torch

# Define the CNN Model
class PotatoCNNModel(nn.Module):
    def __init__(self):
        super(PotatoCNNModel, self).__init__()

        # Convolutional Layers
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.conv5 = nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding=1)

        #Batch norm
        self.batch1 = nn.BatchNorm2d(16)
        self.batch2 = nn.BatchNorm2d(32)
        self.batch3 = nn.BatchNorm2d(64)
        self.batch4 = nn.BatchNorm2d(128)
        self.batch5 = nn.BatchNorm2d(256)
        # Activation
        self.relu = nn.ReLU()

        # Pooling Layer
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Fully Connected Layers
        self.fc1 = nn.Linear(256, 128)
        self.fc2 = nn.Linear(128, 3)

    def forward(self, x):
        # Convolution + ReLU + Pooling
        x = self.pool(self.relu(self.batch1(self.conv1(x))))
        x = self.pool(self.relu(self.batch2(self.conv2(x))))
        x = self.pool(self.relu(self.batch3(self.conv3(x))))
        x = self.pool(self.relu(self.batch4(self.conv4(x))))
        x = self.pool(self.relu(self.batch5(self.conv5(x))))

        # Flatten
        x = x.view(x.size(0), -1)

        # Fully Connected Layers
        x = self.relu(self.fc1(x))
        x = self.fc2(x)  # Logits

        return x                # Although this is a classification problem, we didn't apply softmax. Do you know why?👀 (Hint: CrossEntropyLoss has something to do here👀)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # it is better to run CNNs with GPUs for faster computation
model = PotatoCNNModel().to(device)

In [ ]:
from tqdm import tqdm    # Shows progress bar

# 🔹 Training Loop
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(dataloader):
        images, labels = images.to(device), labels.to(device)


        outputs = model(images).squeeze()  # The model outputs in shape [batch_size,1]. We convert them to [batch_size,] so the loss accepts them.
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # Track accuracy
        predictions = torch.softmax(outputs, dim=1)
        predictions = torch.argmax(predictions, dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy


def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(dataloader):
        images, labels = images.to(device), labels.to(device)


        outputs = model(images).squeeze()  # The model outputs in shape [batch_size,1]. We convert them to [batch_size,] so the loss accepts them.
        loss = criterion(outputs, labels)
        total_loss += loss.item()

        # Track accuracy
        predictions = torch.softmax(outputs, dim=1)
        predictions = torch.argmax(predictions, dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

# def validate(model, dataloader, criterion, device):
#     model.eval()  # Set model to evaluation mode, you will understand why later
#     total_loss = 0
#     correct = 0
#     total = 0

#     with torch.no_grad():  # Disable gradient calculation
#         for images, labels in tqdm(dataloader):
#             images, labels = images.to(device), labels.to(device)

#             outputs = model(images)  # Forward pass
#             loss = criterion(outputs, labels)  # Compute loss
#             total_loss += loss.item()

#             # (Optional) Compute accuracy
#             predictions = outputs.argmax(dim=1)  # Get class with highest probability
#             correct += (predictions == labels).sum().item()
#             total += labels.size(0)

#     avg_loss = total_loss / len(dataloader)
#     accuracy = 100 * correct / total  # Compute accuracy in percentage
#     return avg_loss, accuracy


In [ ]:
import torch.optim as optim

# Training setup
criterion = nn.CrossEntropyLoss()

# TO-DO: Set learning rate
learning_rate = 0.001

# TO-DO: Define optimizer
# HINT: Pass model.parameters() and learning_rate
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# TO-DO: Set number of epochs
num_epochs = 10  # How many times to iterate through the dataset?

# Initialize history tracking
history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}

# Training loop
for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion,optimizer,device)
    test_loss, test_acc = validate(model, test_loader, criterion,device)

    # Store history
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["test_loss"].append(test_loss)
    history["test_acc"].append(test_acc)

    print(f'Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}')


In [ ]:
# Write your code here
import torch.nn as nn
import torch

# Define the CNN Model
class PotatoCNNModel(nn.Module):
    def __init__(self):
        super(PotatoCNNModel, self).__init__()

        # Convolutional Layers
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.conv5 = nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding=1)

        #Batch norm
        self.batch1 = nn.BatchNorm2d(16)
        self.batch2 = nn.BatchNorm2d(32)
        self.batch3 = nn.BatchNorm2d(64)
        self.batch4 = nn.BatchNorm2d(128)
        self.batch5 = nn.BatchNorm2d(256)
        # Activation
        self.relu = nn.ReLU()

        # Pooling Layer
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Fully Connected Layers
        self.fc1 = nn.Linear(256, 128)
        self.fc2 = nn.Linear(128, 3)

    def forward(self, x):
        # Convolution + ReLU + Pooling
        x = self.pool(self.relu(self.batch1(self.conv1(x))))
        x = self.pool(self.relu(self.batch2(self.conv2(x))))
        x = self.pool(self.relu(self.batch3(self.conv3(x))))
        x = torch.cat([x, self.conv2], dim=1)
        x = self.pool(self.relu(self.batch4(self.conv4(x))))
        x = self.pool(self.relu(self.batch5(self.conv5(x))))

        # Flatten
        x = x.view(x.size(0), -1)

        # Fully Connected Layers
        x = self.relu(self.fc1(x))
        x = self.fc2(x)  # Logits

        return x                # Although this is a classification problem, we didn't apply softmax. Do you know why?👀 (Hint: CrossEntropyLoss has something to do here👀)

In [ ]:
import torch.optim as optim

# Training setup
criterion = nn.CrossEntropyLoss()

# TO-DO: Set learning rate
learning_rate = 0.001

# TO-DO: Define optimizer
# HINT: Pass model.parameters() and learning_rate
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# TO-DO: Set number of epochs
num_epochs = 10  # How many times to iterate through the dataset?

# Initialize history tracking
history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}

# Training loop
for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion,optimizer,device)
    test_loss, test_acc = validate(model, test_loader, criterion,device)

    # Store history
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["test_loss"].append(test_loss)
    history["test_acc"].append(test_acc)

    print(f'Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}')
